In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/DeogenesMaranan/ngiml"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/ngiml")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run([
        "git",
        "clone",
        "--branch",
        REPO_BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
print(f"Repo ready at {REPO_DIR} on branch {REPO_BRANCH}")

In [2]:
import os
from pathlib import Path
from huggingface_hub import login, snapshot_download
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
DATASET_REPO = "juhenes/ngiml"
DATASET_REVISION = "main"
DATA_DIR = "/content/data"

if HF_TOKEN:
    login(token=HF_TOKEN)

os.makedirs(DATA_DIR, exist_ok=True)
snapshot_download(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    local_dir=DATA_DIR,
    revision=DATASET_REVISION,
    token=HF_TOKEN,
    resume_download=True,
    allow_patterns=[
        "CASIA1/*", "CASIA2/*", "Columbia/*", "COVERAGE/*",
        "manifest.*"
        # "TampCOCO/*"
    ],
)

root = Path(DATA_DIR)
manifest_files = sorted(
    p for p in root.rglob("manifest.*")
    if p.name in {"manifest.parquet", "manifest.json"}
    and any(parent.name in ["CASIA1", "CASIA2", "Columbia", "COVERAGE"] for parent in p.parents)
    # "TampCOCO/*"
)
tar_count = sum(1 for _ in root.rglob("*.tar")) + sum(1 for _ in root.rglob("*.tar.gz")) + sum(1 for _ in root.rglob("*.tgz"))

print("Dataset ready at", DATA_DIR)
print("Found manifests:", [str(p) for p in manifest_files[:5]])
print("Tar shards count:", tar_count)

In [3]:
from google.colab import drive
from pathlib import Path

DRIVE_MOUNT = "/content/drive"
OUTPUT_DIR = f"{DRIVE_MOUNT}/MyDrive/ngiml_runs"

drive.mount(DRIVE_MOUNT)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Checkpoints will be written to", OUTPUT_DIR)


In [4]:
from pathlib import Path
import json
from collections import Counter

from tools.manifest_utils import find_or_resolve_manifest
from src.data.config import Manifest
from src.data.dataloaders import load_manifest

SELECTED_TRAIN_VAL_DATASETS = ["CASIA2"]
# "TampCOCO"

data_root = Path(DATA_DIR)
MANIFEST_PATH = find_or_resolve_manifest(data_root)

manifest_obj = load_manifest(MANIFEST_PATH)

train_samples = [
    s for s in manifest_obj.samples
    if s.split == "train" and str(getattr(s, "dataset", "")).upper() in [d.upper() for d in SELECTED_TRAIN_VAL_DATASETS]
]
val_samples = [
    s for s in manifest_obj.samples
    if s.split == "val" and str(getattr(s, "dataset", "")).upper() in [d.upper() for d in SELECTED_TRAIN_VAL_DATASETS]
]

test_samples = [s for s in manifest_obj.samples if s.split == "test"]

filtered_samples = train_samples + val_samples + test_samples
filtered_manifest = Manifest(
    samples=filtered_samples,
    normalization_mode=manifest_obj.normalization_mode
)

FILTERED_MANIFEST_PATH = data_root / 'manifest_filtered.json'
with open(FILTERED_MANIFEST_PATH, 'w', encoding='utf-8') as handle:
    json.dump(filtered_manifest.to_dict(), handle, indent=2)
MANIFEST_PATH = FILTERED_MANIFEST_PATH

train_val_samples = train_samples + val_samples
train_val_split_counts = Counter(s.split for s in train_val_samples)
train_val_dataset_counts = Counter(s.dataset for s in train_val_samples)

print('Using filtered manifest:', MANIFEST_PATH)
print('Train/Val split counts:', dict(train_val_split_counts))
print('Train/Val dataset counts:', dict(train_val_dataset_counts))
print(f'Train samples: {len(train_samples)}, Val samples: {len(val_samples)}, Test samples: {len(test_samples)}')

In [5]:
import json
import dataclasses

from tools.train_ngiml import TrainConfig
from tools.colab_train_helpers import (
    apply_colab_runtime_settings,
    apply_phase2_resume_preset,
    stage_persistent_cache_to_runtime,
)

PERSISTENT_CACHE_DIR = f"{OUTPUT_DIR}/local_cache"
RUNTIME_CACHE_DIR = "/content/cache"
RUNTIME_CACHE_DIR = "/content/cache"
stage_info = stage_persistent_cache_to_runtime(
    persistent_cache_dir=PERSISTENT_CACHE_DIR,
    runtime_cache_dir=RUNTIME_CACHE_DIR,
    force=False,
)
print("Cache staging:", stage_info)

training_config = TrainConfig(
    manifest=MANIFEST_PATH,
    scheduler_type="cosine",
    output_dir=OUTPUT_DIR,
    batch_size=20,
    epochs=50,
    num_workers=6,
    amp=True,
    pin_memory=True,
    channels_last=True,
    compile_model=True,
    compile_mode="default",
    deterministic=False,
    use_tf32=True,
    precision="bf16",
    gradient_checkpointing=True,
    flash_attention=True,
    xformers=True,
    cuda_expandable_segments=True,
    lr_schedule=True,
    warmup_epochs=3,
    min_lr_scale=0.1,
    grad_clip=1.0,
    grad_accum_steps=1,
    val_every=1,
    checkpoint_every=1,
    resume=None,
    auto_resume=True,
    round_robin_seed=42,
    balance_sampling=False,
    balance_real_fake=False,
    balanced_positive_ratio=0.5,
    balanced_sampler_seed=42,
    balanced_sampler_num_samples=None,
    prefetch_factor=2,
    persistent_workers=False,
    drop_last=True,
    auto_local_cache=True,
    local_cache_dir=RUNTIME_CACHE_DIR,
    reuse_local_cache_manifest=True,
    views_per_sample=3,
    max_short_side=480,
    max_rotation_degrees=6.0,
    noise_std_max=0.012,
    disable_aug=False,
    device="cuda",
    aug_seed=42,
    seed=42,
    early_stopping_patience=12,
    early_stopping_min_delta=0.0001,
    early_stopping_monitor="f1",
    training_phase="phase1",
    auto_phase2_enabled=True,
    auto_phase2_patience=5,
    auto_phase2_lr_scale=0.33,
    auto_phase2_tversky_weight=0.1,
    auto_phase2_monitor="iou",
    metric_threshold=0.5,
    optimize_threshold=True,
    threshold_metric="f1",
    threshold_start=0.2,
    threshold_end=0.8,
    threshold_step=0.02,
    small_mask_ratio_max=0.01,
    medium_mask_ratio_max=0.05,
    compute_foreground_ratio=True,
    foreground_ratio_max_batches=20,
    short_side_probe_samples=0,
    auto_pos_weight=True,
    pos_weight_min=0.5,
    pos_weight_max=10.0,
    balanced_pos_weight_cap=0.0,
    loss_hybrid_mode="dice_bce",
    dice_weight=1.0,
    bce_weight=1.0,
    focal_gamma=2.0,
    focal_alpha=0.25,
    tversky_weight=0.0,
    tversky_alpha=0.3,
    tversky_beta=0.8,
    lovasz_weight=0.0,
    use_boundary_loss=True,
    boundary_weight=0.03,
    ema_enabled=True,
    ema_decay=0.999,
    hard_mining_enabled=False,
    hard_mining_start_epoch=5,
    hard_mining_weight=0.03,
    hard_mining_gamma=2.0,
    default_aug={
        "enable": True,
        "views_per_sample": 3,
        "enable_flips": True,
        "enable_rotations": True,
        "max_rotation_degrees": 6.0,
        "enable_random_crop": True,
        "crop_scale_range": [0.75, 1.0],
        "object_crop_bias_prob": 0.85,
        "min_fg_pixels_for_object_crop": 8,
        "enable_elastic": False,
        "elastic_prob": 0.0,
        "elastic_alpha": 8.0,
        "elastic_sigma": 5.0,
        "enable_color_jitter": True,
        "color_jitter_factors": [0.9, 1.1],
        "brightness_jitter_factors": [0.9, 1.1],
        "contrast_jitter_factors": [0.9, 1.1],
        "enable_noise": True,
        "noise_std_range": [0.0, 0.012],
        "multiscale_training": False,
        "multiscale_short_side_range": [384, 640],
    },
    per_dataset_aug={},
    model_config={
        "efficientnet": {
            "pretrained": True,
            "out_indices": [1, 2, 3, 4],
            "enforce_input_size": False,
            "input_size": None,
        },
        "swin": {
            "model_name": "swin_tiny_patch4_window7_224",
            "pretrained": True,
            "out_indices": [0, 1, 2, 3],
            "input_size": 384,
        },
        "residual": {
            "num_kernels": 3,
            "base_channels": 32,
            "num_stages": 4,
        },
        "fusion": {
            "fusion_channels": [64, 128, 192, 256],
            "noise_branch": "residual",
            "noise_skip_stage": None,
            "noise_decay": 1.0,
            "norm": "bn",
            "activation": "relu",
            "fusion_refinement": True,
        },
        "decoder": {
            "decoder_channels": None,
            "out_channels": 1,
            "norm": "in",
            "activation": "relu",
            "per_stage_heads": True,
            "enable_edge_guidance": True,
            "use_dropout": True,
            "dropout_p": 0.2,
        },
        "optimizer": {
            "efficientnet": {
                "lr": 1.2909944487358057e-05,
                "weight_decay": 0.0001704329049701249,
            },
            "swin": {
                "lr": 6.4549722436790284e-06,
                "weight_decay": 0.00011362193664674994,
            },
            "residual": {
                "lr": 0.0003227486121839514,
                "weight_decay": 0.00022724387329349987,
            },
            "fusion": {
                "lr": 0.0001549193338482967,
                "weight_decay": 0.00022724387329349987,
            },
            "decoder": {
                "lr": 0.00023237900077244502,
                "weight_decay": 0.00022724387329349987,
            },
            "betas": [0.9, 0.999],
            "eps": 1e-08,
            "freeze_backbone_epochs": 3,
        },
        "use_low_level": True,
        "use_context": True,
        "use_residual": True,
        "enable_residual_attention": True,
        "gradient_checkpointing": True,
        "flash_attention": True,
        "xformers": True,
    },
    loss_config={
        "dice_weight": 1.0,
        "bce_weight": 1.0,
        "pos_weight": 1.0,
        "stage_weights": [0.05, 0.1, 0.2, 1.0],
        "smooth": 1e-06,
        "hybrid_mode": "dice_bce",
        "focal_gamma": 2.0,
        "focal_alpha": 0.25,
        "tversky_weight": 0.0,
        "tversky_alpha": 0.3,
        "tversky_beta": 0.8,
        "lovasz_weight": 0.0,
        "use_boundary_loss": True,
        "boundary_weight": 0.03,
        "hard_pixel_mining": False,
    },
    debug_timing=False,
)

print("Configuration successfully loaded.")

BALANCE_SAMPLING = False
TUNE_FOR_LARGE_BATCH = True
training_config = apply_colab_runtime_settings(
    training_config,
    balance_sampling=BALANCE_SAMPLING,
    tune_for_large_batch=TUNE_FOR_LARGE_BATCH,
    local_cache_dir=RUNTIME_CACHE_DIR,
 )

USE_PHASE2_RESUME = False
PHASE2_RESUME_CHECKPOINT = f"{OUTPUT_DIR}/checkpoints/best_iou_checkpoint.pt"
if USE_PHASE2_RESUME:
    training_config = apply_phase2_resume_preset(
        training_config,
        resume_checkpoint=PHASE2_RESUME_CHECKPOINT,
        lr_scale=0.33,
        tversky_weight=0.1,
        monitor_metric="iou",
    )
    print("Applied phase-2 resume preset:", PHASE2_RESUME_CHECKPOINT)

effective_view_multiplier = {
    name: cfg.views_per_sample if cfg.enable else 1
    for name, cfg in training_config.get("per_dataset_aug", {}).items()
}

print("Applied runtime settings:")
summary_keys = [
    "batch_size",
    "num_workers",
    "persistent_workers",
    "pin_memory",
    "auto_local_cache",
    "local_cache_dir",
    "reuse_local_cache_manifest",
    "compile_model",
    "compile_mode",
    "channels_last",
    "use_tf32",
    "balance_sampling",
    "balance_real_fake",
    "balanced_positive_ratio",
    "balanced_pos_weight_cap",
    "resume",
    "auto_resume",
    "threshold_metric",
    "early_stopping_monitor",
    "tversky_weight",
]
print({k: training_config.get(k) for k in summary_keys})
print("Per-dataset views_per_sample:", effective_view_multiplier)

print("Effective training config (post-settings):")
print(json.dumps(
    training_config,
    indent=2,
    default=lambda o: dataclasses.asdict(o) if dataclasses.is_dataclass(o) and not isinstance(o, type) else str(o)
))

In [ ]:
from importlib import reload
from tools import train_ngiml

reload(train_ngiml)

import importlib
try:
    import src.data.dataloaders as dl
    importlib.reload(dl)
except Exception:
    pass

from tools.train_ngiml import AugmentationConfig
if isinstance(training_config.get("default_aug"), type):
    training_config["default_aug"] = AugmentationConfig()
if isinstance(training_config.get("per_dataset_aug"), dict):
    for k, v in training_config["per_dataset_aug"].items():
        if isinstance(v, type):
            training_config["per_dataset_aug"][k] = AugmentationConfig()

cfg = train_ngiml.TrainConfig(**training_config)
train_ngiml.run_training(cfg)

In [ ]:
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from tools.manifest_utils import find_or_resolve_manifest
from tools.infer_helpers import (
    collate_eval_batch_like_training,
    load_model_from_checkpoint,
    normalize_image_for_inference,
    should_use_high_pass_for_records,
    )
from src.data.dataloaders import load_manifest

RUN_OUTPUT_DIR = Path(training_config.get("output_dir", OUTPUT_DIR))
CHECKPOINT_DIR = RUN_OUTPUT_DIR / "checkpoints"
if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(f"Checkpoint directory not found: {CHECKPOINT_DIR}")

best_iou_candidates = sorted(CHECKPOINT_DIR.glob("checkpoint_epoch_016.pt"), key=lambda p: p.stat().st_mtime)
if best_iou_candidates:
    CKPT_PATH = best_iou_candidates[-1]
else:
    best_ckpt_candidates = sorted(CHECKPOINT_DIR.glob("best_checkpoint.pt"), key=lambda p: p.stat().st_mtime)
    if best_ckpt_candidates:
        CKPT_PATH = best_ckpt_candidates[-1]
    else:
        ckpt_candidates = sorted(CHECKPOINT_DIR.glob("checkpoint_epoch_*.pt"), key=lambda p: p.stat().st_mtime)
        if not ckpt_candidates:
            raise FileNotFoundError(f"No checkpoint found under {CHECKPOINT_DIR}")
        CKPT_PATH = ckpt_candidates[-1]

MANIFEST_PATH = find_or_resolve_manifest(Path(DATA_DIR))
manifest = load_manifest(MANIFEST_PATH)
INFER_NORMALIZATION = str(getattr(manifest, "normalization_mode", "zero_one")).strip().lower()

model, device, ckpt_info = load_model_from_checkpoint(CKPT_PATH)
VAL_THRESHOLD = float(ckpt_info.get("default_threshold", 0.5))
INFER_MAX_SHORT_SIDE = int(ckpt_info.get("max_short_side", 0) or 0)
VAL_THRESHOLD_SOURCE = str(ckpt_info.get("threshold_source", "unknown"))

threshold_metric_name = str(training_config.get("threshold_metric", "dice")).strip().lower()
threshold_start = float(training_config.get("threshold_start", 0.1))
threshold_end = float(training_config.get("threshold_end", 0.8))
threshold_step = float(training_config.get("threshold_step", 0.02))
if threshold_step <= 0.0:
    raise ValueError(f"threshold_step must be positive, got {threshold_step}")
thresholds = np.round(np.arange(threshold_start, threshold_end + 0.5 * threshold_step, threshold_step), 6)
if thresholds.size == 0:
    raise RuntimeError("Threshold sweep grid is empty. Check threshold_start, threshold_end, and threshold_step.")

def _metrics_from_counts(tp, tn, fp, fn, eps=1e-6):
    iou = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    f1 = (2.0 * precision * recall) / (precision + recall + eps)
    dice = (2.0 * tp + eps) / (2.0 * tp + fp + fn + eps)
    accuracy = (tp + tn + eps) / (tp + tn + fp + fn + eps)
    return {
        "dice": float(dice),
        "iou": float(iou),
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "accuracy": float(accuracy),
    }

def _evaluate_threshold(probabilities, targets, threshold):
    tp = tn = fp = fn = 0.0
    for pred_prob, target in zip(probabilities, targets):
        pred = (pred_prob >= threshold).astype(np.float32)
        tp += float((pred * target).sum())
        tn += float(((1.0 - pred) * (1.0 - target)).sum())
        fp += float((pred * (1.0 - target)).sum())
        fn += float(((1.0 - pred) * target).sum())
    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        **_metrics_from_counts(tp, tn, fp, fn),
    }

def _sweep_thresholds(probabilities, targets):
    rows = []
    best_row = None
    best_score = None
    for threshold in thresholds:
        row = {
            "threshold": float(threshold),
            **_evaluate_threshold(probabilities, targets, float(threshold)),
        }
        metric_value = float(row.get(threshold_metric_name, row["dice"]))
        if (
            best_row is None
            or metric_value > best_score + 1e-12
            or (abs(metric_value - best_score) <= 1e-12 and row["dice"] > best_row["dice"] + 1e-12)
            or (
                abs(metric_value - best_score) <= 1e-12
                and abs(row["dice"] - best_row["dice"]) <= 1e-12
                and row["threshold"] < best_row["threshold"]
            )
        ):
            best_row = row
            best_score = metric_value
        rows.append(row)
    return rows, best_row

def _confusion_from_totals(tn, fp, fn, tp):
    return np.array([[tn, fp], [fn, tp]], dtype=np.float64)

TEST_SPLIT = "test"
EVAL_BATCH_SIZE = max(1, int(training_config.get("batch_size", 1)))
test_samples = [sample for sample in manifest.samples if sample.split == TEST_SPLIT]
if not test_samples:
    raise RuntimeError(f"No samples found for split={TEST_SPLIT!r} in manifest {MANIFEST_PATH}")

manifest_mask_paths = sum(1 for sample in test_samples if sample.mask_path is not None)
positive_labeled_samples = sum(1 for sample in test_samples if int(getattr(sample, "label", 0)) == 1)
dataset_counts = {}
for sample in test_samples:
    dataset_counts[sample.dataset] = dataset_counts.get(sample.dataset, 0) + 1

USE_HIGH_PASS_IN_EVAL = should_use_high_pass_for_records(test_samples)

print("Using checkpoint:", CKPT_PATH)
print("Using manifest:", MANIFEST_PATH)
print(f"Evaluating split: {TEST_SPLIT}")
print(f"Samples: {len(test_samples)} | manifest mask paths: {manifest_mask_paths} | positive labels: {positive_labeled_samples}")
print("Dataset counts:", dataset_counts)
print(f"Validation threshold from checkpoint: {VAL_THRESHOLD:.4f} ({VAL_THRESHOLD_SOURCE})")
print("Sweep metric:", threshold_metric_name)
print("Sweep threshold range:", [float(thresholds[0]), float(thresholds[-1]), float(threshold_step)])
print("Inference normalization:", INFER_NORMALIZATION)
print("Inference max_short_side:", INFER_MAX_SHORT_SIDE)
print("Inference high_pass enabled:", USE_HIGH_PASS_IN_EVAL)
print("Eval batch size:", EVAL_BATCH_SIZE)

dataset_probabilities = defaultdict(list)
dataset_targets = defaultdict(list)
all_probabilities = []
all_targets = []

model.eval()
progress = tqdm(total=len(test_samples), desc="Preparing predictions", unit="sample")
for start_idx in range(0, len(test_samples), EVAL_BATCH_SIZE):
    batch_records = test_samples[start_idx:start_idx + EVAL_BATCH_SIZE]
    batch_images, batch_masks, batch_high_pass, _ = collate_eval_batch_like_training(
        batch_records,
        max_short_side=INFER_MAX_SHORT_SIDE,
        use_high_pass=USE_HIGH_PASS_IN_EVAL,
    )

    normalized_batch_images = torch.stack(
        [normalize_image_for_inference(image, normalization_mode=INFER_NORMALIZATION) for image in batch_images],
        dim=0,
    ).to(device)

    batch_high_pass_device = None
    if batch_high_pass is not None:
        batch_high_pass_device = batch_high_pass.to(device)

    with torch.no_grad():
        logits = model(
            normalized_batch_images,
            target_size=batch_masks.shape[-2:],
            high_pass=batch_high_pass_device,
        )[-1]
        batch_probs = torch.sigmoid(logits).detach().cpu().numpy()
    batch_targets = (batch_masks[:, 0].cpu().numpy() >= 0.5).astype(np.float32)

    for local_idx, sample in enumerate(batch_records):
        pred_prob = batch_probs[local_idx, 0]
        target = batch_targets[local_idx]

        dataset_key = str(sample.dataset)
        dataset_probabilities[dataset_key].append(pred_prob)
        dataset_targets[dataset_key].append(target)
        all_probabilities.append(pred_prob)
        all_targets.append(target)

    processed = start_idx + len(batch_records)
    progress.update(len(batch_records))
progress.close()

CALIBRATED_TEST_THRESHOLD_TABLE = {}
CALIBRATED_TEST_THRESHOLDS_BY_DATASET = {}

overall_rows, overall_best = _sweep_thresholds(all_probabilities, all_targets)
CALIBRATED_TEST_THRESHOLD = float(overall_best["threshold"])
CALIBRATED_TEST_THRESHOLD_TABLE["__overall__"] = overall_rows

for dataset_name in sorted(dataset_probabilities):
    rows, best_row = _sweep_thresholds(dataset_probabilities[dataset_name], dataset_targets[dataset_name])
    CALIBRATED_TEST_THRESHOLD_TABLE[dataset_name] = rows
    CALIBRATED_TEST_THRESHOLDS_BY_DATASET[dataset_name] = float(best_row["threshold"])

print("Calibration summary:")
print({
    "overall_best_threshold": CALIBRATED_TEST_THRESHOLD,
    "overall_best_metric": float(overall_best.get(threshold_metric_name, overall_best["dice"])),
    "per_dataset_best_thresholds": CALIBRATED_TEST_THRESHOLDS_BY_DATASET,
})

strategies = {
    "val_threshold": {
        "threshold": VAL_THRESHOLD,
        "threshold_source": "checkpoint_val_threshold",
    },
    "swept_threshold": {
        "threshold": CALIBRATED_TEST_THRESHOLD,
        "threshold_source": f"swept_test_{threshold_metric_name}",
    },
}

comparison_rows = []
per_dataset_comparison = {}
for strategy_name, info in strategies.items():
    overall_eval = _evaluate_threshold(all_probabilities, all_targets, float(info["threshold"]))
    per_dataset_eval = {}
    for dataset_name in sorted(dataset_probabilities):
        per_dataset_eval[dataset_name] = _evaluate_threshold(
            dataset_probabilities[dataset_name],
            dataset_targets[dataset_name],
            float(info["threshold"]),
        )

    per_dataset_comparison[strategy_name] = per_dataset_eval
    comparison_rows.append({
        "strategy": strategy_name,
        "threshold": float(info["threshold"]),
        "threshold_source": info["threshold_source"],
        "samples": len(all_probabilities),
        "dice": overall_eval["dice"],
        "iou": overall_eval["iou"],
        "f1": overall_eval["f1"],
        "precision": overall_eval["precision"],
        "recall": overall_eval["recall"],
        "accuracy": overall_eval["accuracy"],
        "tp": overall_eval["tp"],
        "tn": overall_eval["tn"],
        "fp": overall_eval["fp"],
        "fn": overall_eval["fn"],
    })

print("\nOverall strategy comparison:")
for row in comparison_rows:
    print({
        "strategy": row["strategy"],
        "threshold": float(row["threshold"]),
        "threshold_source": row["threshold_source"],
        "dice": float(row["dice"]),
        "iou": float(row["iou"]),
        "precision": float(row["precision"]),
        "recall": float(row["recall"]),
        "accuracy": float(row["accuracy"]),
    })

if len(comparison_rows) == 2:
    base, swept = comparison_rows
    print("\nDelta (swept_threshold - val_threshold):")
    print({
        "dice_delta": float(swept["dice"] - base["dice"]),
        "iou_delta": float(swept["iou"] - base["iou"]),
        "precision_delta": float(swept["precision"] - base["precision"]),
        "recall_delta": float(swept["recall"] - base["recall"]),
        "accuracy_delta": float(swept["accuracy"] - base["accuracy"]),
    })

print("\nPer-dataset strategy comparison:")
for dataset_name in sorted(dataset_probabilities):
    print(f"Dataset: {dataset_name}")
    for strategy_name in strategies:
        row = per_dataset_comparison[strategy_name][dataset_name]
        print({
            "strategy": strategy_name,
            "threshold": float(strategies[strategy_name]["threshold"]),
            "dice": float(row["dice"]),
            "iou": float(row["iou"]),
            "precision": float(row["precision"]),
            "recall": float(row["recall"]),
            "accuracy": float(row["accuracy"]),
        })

heatmaps = []
for row in comparison_rows:
    heatmaps.append((
        f"{row['strategy']}\nthreshold={row['threshold']:.3f}",
        _confusion_from_totals(row["tn"], row["fp"], row["fn"], row["tp"]),
    ))

fig, axes = plt.subplots(1, len(heatmaps), figsize=(6 * len(heatmaps), 4))
if len(heatmaps) == 1:
    axes = [axes]

for ax, (title, confusion_matrix) in zip(axes, heatmaps):
    im = ax.imshow(confusion_matrix, cmap="Blues")
    ax.set_title(f"{title}\nPixel Confusion Matrix")
    ax.set_xticks([0, 1], labels=["Pred Neg", "Pred Pos"])
    ax.set_yticks([0, 1], labels=["True Neg", "True Pos"])
    for row_idx in range(confusion_matrix.shape[0]):
        for col_idx in range(confusion_matrix.shape[1]):
            value = confusion_matrix[row_idx, col_idx]
            ax.text(col_idx, row_idx, f"{value:,.0f}", ha="center", va="center", color="black")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pixel count")

plt.tight_layout()
plt.show()

print("\nSaved globals for later cells:")
print({
    "VAL_THRESHOLD": VAL_THRESHOLD,
    "VAL_THRESHOLD_SOURCE": VAL_THRESHOLD_SOURCE,
    "CALIBRATED_TEST_THRESHOLD": CALIBRATED_TEST_THRESHOLD,
    "CALIBRATED_TEST_THRESHOLDS_BY_DATASET": CALIBRATED_TEST_THRESHOLDS_BY_DATASET,
})

In [ ]:
from pathlib import Path
import json, glob, torch, pandas as pd

CHECKPOINTS_DIR = path(f"{OUTPUT_DIR}/checkpoints/")

def load_json_logs(checkpoints_dir: Path):
    logs = []
    main = checkpoints_dir / "checkpoint_metrics.json"
    candidates = [main] + sorted(checkpoints_dir.glob("checkpoint_metrics.json.corrupt.*"))
    for p in candidates:
        if not p.exists():
            continue
        try:
            with open(p, "r", encoding="utf-8") as fh:
                data = json.load(fh)
            if isinstance(data, dict):
                logs.extend([data])
            elif isinstance(data, list):
                logs.extend(data)
            print(f"Loaded {len(data) if isinstance(data,(list,dict)) else 'N/A'} entries from {p.name}")
            if p == main:
                break
        except Exception as e:
            print(f"Failed to read {p.name}: {e}")
    return logs

def scan_checkpoints(checkpoints_dir: Path):
    records = []
    pattern = str(checkpoints_dir / "checkpoint_epoch_*.pt")
    files = sorted(glob.glob(pattern), key=lambda p: int(Path(p).stem.split("_")[-1]) if Path(p).stem.split("_")[-1].isdigit() else -1)
    for p in files:
        try:
            ck = torch.load(p, map_location="cpu")
        except Exception as e:
            print(f"Skipping unreadable checkpoint {p}: {e}")
            continue
        rec = {
            "checkpoint_path": str(p),
            "epoch": int(ck.get("epoch", -1)) if isinstance(ck.get("epoch", None), (int,float,str)) else None,
            "global_step": int(ck.get("global_step", 0)) if ck.get("global_step", None) is not None else None,
        }
        for k in ("train_loss","val_loss","iou","f1","dice","precision","recall","accuracy"):
            if k in ck:
                rec[k] = ck[k]
        for maybe in ("metrics","eval_metrics","val_metrics"):
            if maybe in ck and isinstance(ck[maybe], dict):
                for k,v in ck[maybe].items():
                    rec.setdefault(k, v)
        records.append(rec)
    return records

json_logs = load_json_logs(CHECKPOINTS_DIR)
if json_logs:
    df = pd.DataFrame(json_logs)
else:
    print("No JSON metrics log found -> scanning checkpoints for embedded info")
    scanned = scan_checkpoints(CHECKPOINTS_DIR)
    df = pd.DataFrame(scanned)

if "epoch" in df.columns:
    df["epoch"] = pd.to_numeric(df["epoch"], errors="coerce").fillna(-1).astype(int)
df = df.sort_values(by=["epoch","global_step"], ascending=[True, True]).reset_index(drop=True)

print(f"Collected {len(df)} records. Preview:")
display(df.head(50))
out_csv = CHECKPOINTS_DIR / "checkpoint_metrics_collected.csv"
df.to_csv(out_csv, index=False)
print("Saved CSV to", out_csv)